In [1]:
import hashlib, json, threading, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import requests

URL  = "https://zenodo.org/records/7574468/files/RAW_seismograms.zip?download=1"
DEST = Path(r"C:\Users\fiza\DAS_Datasets\data\otway_microseismic\RAW_seismograms.zip")
MD5  = "7df5b37bc5e839749bb9e082047a18db"
WORKERS = 6                    # slow ya error aaye to 3-4 karein
CHUNK   = 8 * 1024 * 1024      # 8 MiB ke hisse

PART = DEST.with_name(DEST.name + ".part")
PROG = DEST.with_name(DEST.name + ".progress.json")

# ---- file ka size pata karein, aur check karein ki Range support hai
r = requests.get(URL, headers={"Range": "bytes=0-0"}, stream=True, timeout=60)
r.raise_for_status()
if r.status_code != 206:
    raise SystemExit("Server hisson mein download support nahi karta; browser se download karein.")
total = int(r.headers["Content-Range"].split("/")[-1])
r.close()
n_chunks = -(-total // CHUNK)
print(f"size: {total:,} bytes ({total/1024**2:.1f} MiB), {n_chunks} chunks")

# ---- agar purani adhoori file (curl/browser wali) hai to alag hata dein
if DEST.exists():
    if DEST.stat().st_size == total:
        raise SystemExit("File pehle se poori hai, download ki zaroorat nahi.")
    DEST.rename(DEST.with_name(DEST.name + ".old_partial"))
    print("purani adhoori file '.old_partial' naam se hata di (chahein to delete kar dein)")

# ---- resume ki taiyari
done = set(json.load(open(PROG))) if PROG.exists() and PART.exists() else set()
if not PART.exists() or PART.stat().st_size != total:
    with open(PART, "wb") as f:
        f.truncate(total)
    done = set()
lock = threading.Lock()

def fetch(i):
    start, end = i * CHUNK, min((i + 1) * CHUNK, total) - 1
    for attempt in range(30):
        try:
            with requests.get(URL, headers={"Range": f"bytes={start}-{end}"},
                              stream=True, timeout=(15, 60)) as resp:
                if resp.status_code != 206:
                    raise IOError(f"status {resp.status_code}")
                pos = start
                with open(PART, "r+b") as f:
                    f.seek(start)
                    for blk in resp.iter_content(256 * 1024):
                        f.write(blk)
                        pos += len(blk)
            if pos != end + 1:
                raise IOError("short read")
            return i
        except Exception:
            time.sleep(min(2 * (attempt + 1), 30))
    raise RuntimeError(f"chunk {i} 30 koshishon ke baad bhi fail")

todo = [i for i in range(n_chunks) if i not in done]
print(f"baaki chunks: {len(todo)} / {n_chunks}")
t0 = time.time()
with ThreadPoolExecutor(WORKERS) as ex:
    futs = [ex.submit(fetch, i) for i in todo]
    for k, fut in enumerate(as_completed(futs), 1):
        i = fut.result()
        with lock:
            done.add(i)
            json.dump(sorted(done), open(PROG, "w"))
        if k % 5 == 0 or k == len(todo):
            got = len(done) * CHUNK / 1024**2
            print(f"{len(done)}/{n_chunks} chunks  ~{min(got, total/1024**2):.0f} MiB  "
                  f"{(time.time()-t0)/60:.1f} min")

# ---- verify
h = hashlib.md5()
with open(PART, "rb") as f:
    while b := f.read(1 << 20):
        h.update(b)
if h.hexdigest() == MD5:
    PART.rename(DEST)
    PROG.unlink(missing_ok=True)
    print("MD5 MATCH. File ready:", DEST)
else:
    print("MD5 MISMATCH:", h.hexdigest(), "\n.part file rakhi hai, PROG hata kar dobara chalayein")

size: 734,125,423 bytes (700.1 MiB), 88 chunks
baaki chunks: 88 / 88
5/88 chunks  ~40 MiB  90.6 min
10/88 chunks  ~80 MiB  91.3 min
15/88 chunks  ~120 MiB  92.0 min
20/88 chunks  ~160 MiB  93.2 min
25/88 chunks  ~200 MiB  95.2 min
30/88 chunks  ~240 MiB  96.4 min
35/88 chunks  ~280 MiB  97.7 min
40/88 chunks  ~320 MiB  98.4 min
45/88 chunks  ~360 MiB  99.0 min
50/88 chunks  ~400 MiB  100.1 min
55/88 chunks  ~440 MiB  100.9 min
60/88 chunks  ~480 MiB  102.5 min
65/88 chunks  ~520 MiB  103.1 min
70/88 chunks  ~560 MiB  104.1 min
75/88 chunks  ~600 MiB  104.9 min
80/88 chunks  ~640 MiB  105.6 min
85/88 chunks  ~680 MiB  106.4 min
88/88 chunks  ~700 MiB  106.5 min
MD5 MATCH. File ready: C:\Users\fiza\DAS_Datasets\data\otway_microseismic\RAW_seismograms.zip
